<a href="https://colab.research.google.com/github/polizzilab/LASErMPNN/blob/main/run_lasermpnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1) Install LASErMPNN & Dependencies (Hit `Run all` -> Notebook will prompt for restart or crash. This is expected. -> Hit `Run all` again after restart to complete the install)

In [ ]:
# @title ### Install LASErMPNN and Boltz dependencies as separate python installation (~2 min)
import os
import re
import subprocess
import sys

venv_dir = ".venv"
venv_py = os.path.join(venv_dir, "bin", "python")

# 1) Create virtualenv if needed
if not os.path.isdir(venv_dir):
    subprocess.check_call(["uv", "venv", venv_dir])

print("Venv python:", venv_py)

# 2) Install base stack into the venv with uv, including pip itself
env = os.environ.copy()
env["UV_PYTHON"] = venv_py

subprocess.check_call(
    [
        "uv", "pip", "install",
        "pip",                # put pip into the venv
        "torch==2.8.0",
        "numpy<2.0.0",
        "scipy",
        "pandas",
        "scikit-learn",
        "h5py",
        "pytest",
        "pydssp",
        "prody",
        "matplotlib",
        "seaborn",
        "jupyter",
        "plotly",
        "pykeops",
        "logomaker",
        "wandb",
        "tqdm",
        "rdkit",
        "py3Dmol",
        "openbabel-wheel",
        "boltz[cuda]==2.2.1",
        "freesasa",
    ],
    env=env,
)

# 3) Inside the venv: detect torch CUDA + install matching PyG wheels with pip -f
pyg_install_code = r"""
import re
import sys
import subprocess
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA runtime version:", torch.version.cuda)

m = re.match(r"^(\d+\.\d+\.\d+)(\+cu\d+)?", torch.__version__)
if not m:
    raise SystemExit(f"Unexpected torch version: {torch.__version__}")

base_version, cuda_suffix = m.groups()
cuda_suffix = cuda_suffix or ""

if cuda_suffix:
    pyg_url = f"https://data.pyg.org/whl/torch-{base_version}{cuda_suffix}.html"
else:
    pyg_url = f"https://data.pyg.org/whl/torch-{base_version}+cpu.html"

print("Using PyG wheel index:", pyg_url)

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "torch_scatter",
    "torch_cluster",
    "-f", pyg_url,
])

print("All installs completed in venv:", sys.executable)
"""

subprocess.check_call([venv_py, "-c", pyg_install_code])

print("Done. Use this Python:", venv_py)

!./.venv/bin/python -c "import torch, torch_scatter; print(torch.__version__); print(torch.cuda.is_available())"

In [ ]:
# @title
!git clone https://github.com/polizzilab/LASErMPNN.git --depth 1 2> /dev/null
%pip install prody py3Dmol

# 2) Use LASErMPNN to Design Sequences

In [ ]:
# @title Upload an input PDB file to run LASErMPNN on. (See upload prompt below cell after executing)
import os
from google.colab import files
import prody as pr
from pathlib import Path
import shutil

residue_one_letter_codes = {'C', 'D', 'S', 'Q', 'K', 'I', 'P', 'T', 'F', 'N', 'G', 'H', 'L', 'R', 'W', 'A', 'V', 'E', 'Y', 'M', 'X'}

# @markdown # Use [ProDy selection syntax](http://www.bahargroup.org/prody/manual/reference/atomic/select.html) to restrict design to specific residues or chains.
# @markdown - #### To design all protein residues, enter `protein` or leave empty.
# @markdown - #### To restrict design to a single chain (chain A), enter `chid A`
# @markdown - #### To restrict design to two chains (chain A and chain B), enter `(chid A) or (chid B)`
# @markdown - #### To restrict design to certain residue indices (residx 1 to 5), enter `resindex 1 2 3 4 5` or `resindex 1 to 5`
# @markdown - #### To restrict design to region around the ligand, enter `protein within 5.0 of (not protein)`

prody_selection_string = "" # @param {"type":"string","placeholder":"protein"}
if prody_selection_string == '' or prody_selection_string is None:
  prody_selection_string = 'protein'

# @markdown # Comma separated list of amino acid one-letter-codes to prevent LASErMPNN from sampling, ex: Cysteine `C`
disable_residues = "" # @param {"type":"string","placeholder":""}
if disable_residues is None:
  disable_residues = ""

disabled_residues = ['X']
for residue in disable_residues.split(','):
  if residue == '':
    continue
  if len(residue) != 1 or residue not in residue_one_letter_codes:
    raise ValueError(f'Invalid residue in disable_residues: {residue}')
  disabled_residues.append(residue)
disabled_residues_string = ','.join(disabled_residues)

def upload_files():
  upload_dict = files.upload()

  # Get the path to what was uploaded
  pdb_string = upload_dict[list(upload_dict.keys())[0]]
  ipath = (Path('/content/') / list(upload_dict.keys())[0]).absolute()

  # Figure out what type of file it is.
  tmp_opath = Path('/content/temp')
  for suffix in ipath.suffixes:
    tmp_opath = tmp_opath.with_suffix(suffix)

  if ''.join(ipath.suffixes) not in ['.pdb', '.cif', '.pdb.gz', '.cif.gz']:
    raise ValueError(f'File type not supported: {"".join(ipath.suffixes)}')

  shutil.copy(ipath, tmp_opath)
  os.remove(str(ipath.absolute()))

  return tmp_opath

pdb_path = upload_files()

print("Upload a .pdb, .pdb.gz, .cif, or .cif.gz file encoding your protein complex below.")
print(Path(pdb_path).absolute().suffixes)

if '.cif' in Path(pdb_path).suffixes:
  protein = pr.parseMMCIF(str(pdb_path))
else:
  protein = pr.parsePDB(str(pdb_path))

protein.setBetas(1.0)
protein.select(f'same residue as ({prody_selection_string})').setBetas(0.0)
opath = Path(pdb_path)
while opath.suffixes:
  opath = opath.with_suffix('')
pdb_path = opath.with_suffix('.pdb')
pr.writePDB(str(pdb_path), protein)

print(pdb_path)


In [ ]:
# @title Run LASErMPNN on a target file.
import subprocess
import shutil
from pathlib import Path

###########
### Define parameters
num_designs = 15 #@param {type: "integer"}
num_designs = int(num_designs)

sequence_sampling_temp = 0.1 #@param ['0.000001', '0.1', '0.2', '0.3', '0.5'] {type:"raw"}
###########


if Path('/content/output').exists():
  shutil.rmtree('/content/output')

command_string = f'cd /content/; ./.venv/bin/python -m LASErMPNN.run_batch_inference {pdb_path.absolute()} /content/output/ {num_designs} --output_fasta --sequence_temp {sequence_sampling_temp} -d cuda:0 --fix_beta --repack_all --disabled_residues {disabled_residues_string}'
print(command_string)

out = subprocess.run(
    command_string, shell=True, capture_output=True
)

print(out.stderr.decode('utf-8'))
print(out.stdout.decode('utf-8'))

with open('/content/output/designs.fasta', 'r') as f:
  print(f.read())

In [ ]:
# @title Visualize Generated Designs
import py3Dmol
from pathlib import Path

path_options = sorted(list(Path('/content/output/').glob('*.pdb')))
# @markdown # Select a design index from `0` to `num_designs - 1`.
design_to_visualize = 0 # @param {"type":"integer","placeholder":"0"}
assert design_to_visualize < len(path_options)

path_to_visualize = f'/content/output/design_{design_to_visualize}.pdb'
print('visualizing', path_to_visualize)

view = py3Dmol.view(width=1000)
view.addModel(open(path_to_visualize, 'r').read(),'pdb', keepH=True)
view.setBackgroundColor('grey')

view.setStyle({}, {'cartoon': {'color':'green'}})
# view.setStyle({'chain':'C'}, {'cartoon': {'color':'green'}})


view.addStyle({}, {'stick': {'colorscheme': 'greyCarbon'}})
# view.addStyle({'chain':'C'}, {'stick': {'colorscheme': 'greyCarbon'}})

# view.addStyle({'within':{'distance':'8.0', 'sel':{'resn':'GG2'}}}, {'stick': {'colorscheme': 'greenCarbon'}})
view.addStyle({'resn':'GG2'}, {'stick': {'colorscheme':'cyanCarbon'}})
view.zoomTo()
view.show()

In [ ]:
# @title Download LASErMPNN Outputs.
import os

# @markdown ### Select what to download.
download_pdbs = True # @param {"type": "boolean"}
download_fasta = True # @param {"type": "boolean"}

default_zip_output = '/content/lasermpnn_outputs.zip'

if os.path.exists(default_zip_output):
  os.remove(default_zip_output)

if download_pdbs:
  subprocess.run(f'zip -j {default_zip_output} /content/output/*.pdb', shell=True)
  files.download(default_zip_output)

if download_fasta:
  if download_pdbs:
    subprocess.run(f'zip -j {default_zip_output} /content/output/*.fasta', shell=True)
  else:
    files.download('/content/output/designs.fasta')


# 3) Use LASErMPNN to Proofread a Sequence

In [ ]:
# @title Upload an input PDB file to run LASErMPNN Proofreading on. (See upload prompt below cell after executing)
import os
from google.colab import files
import prody as pr
from pathlib import Path
import shutil

residue_one_letter_codes = {'C', 'D', 'S', 'Q', 'K', 'I', 'P', 'T', 'F', 'N', 'G', 'H', 'L', 'R', 'W', 'A', 'V', 'E', 'Y', 'M', 'X'}


def upload_files():
  upload_dict = files.upload()

  # Get the path to what was uploaded
  pdb_string = upload_dict[list(upload_dict.keys())[0]]
  ipath = (Path('/content/') / list(upload_dict.keys())[0]).absolute()

  # Figure out what type of file it is.
  tmp_opath = Path('/content/temp')
  for suffix in ipath.suffixes:
    tmp_opath = tmp_opath.with_suffix(suffix)

  if ''.join(ipath.suffixes) not in ['.pdb', '.cif', '.pdb.gz', '.cif.gz']:
    raise ValueError(f'File type not supported: {"".join(ipath.suffixes)}')

  shutil.copy(ipath, tmp_opath)
  os.remove(str(ipath.absolute()))

  return tmp_opath

pdb_path = upload_files()

print("Upload a .pdb, .pdb.gz, .cif, or .cif.gz file encoding your protein complex below.")
print(Path(pdb_path).absolute().suffixes)

if '.cif' in Path(pdb_path).suffixes:
  protein = pr.parseMMCIF(str(pdb_path))
else:
  protein = pr.parsePDB(str(pdb_path))

opath = Path(pdb_path)
while opath.suffixes:
  opath = opath.with_suffix('')
pdb_path = opath.with_suffix('.pdb')
pr.writePDB(str(pdb_path), protein)

print(pdb_path)

In [ ]:
# @title Run LASErMPNN on a target file (~6 mins)
import subprocess

###########
### Define parameters

# parser.add_argument('--n_decoding_orders', type=int, default=10, help='')
# parser.add_argument('--n_dropouts', type=int, default=10, help='')
# parser.add_argument('--repack_all', action='store_true', help='')
# parser.add_argument('--selection_string', '-r', type=str, default='', help='A Prody selection string to override the default behavior of proofreading the binding site only.')

# @markdown Number of decoding orders to use (default: 10). More is more rigorous, but takes longer to run.
num_dec_orders = 10 #@param {type: "integer"}
num_dec_orders = int(num_dec_orders)

# @markdown Number of dropouts to use (default: 5). More is more rigorous, but takes longer to run.
num_dropouts = 5 #@param {type: "integer"}
num_dropouts = int(num_dropouts)

# @markdown Repack all residues rather than using the input sidechain coordinates as part of the information to condition proofreading on.
repack_all = False #@param {type: "boolean"}

# @markdown Residues to proofread, defaults to first shell around the ligand if left blank. (ProDy format selection string)
prody_selection_string = '' #@param {type: "string"}


###########

if Path('/content/proofreading_out').exists():
  shutil.rmtree('/content/proofreading_out')

command_string = (
    f'cd /content/; ./.venv/bin/python -m LASErMPNN.run_proofreading {pdb_path.absolute()} ./proofreading_out/ --n_decoding_orders {num_dec_orders} --n_dropouts {num_dropouts} --selection_string \'{prody_selection_string}\' -d cuda:0'
    + (' --repack_all' if repack_all else '' )
)
print(command_string)

subprocess.run(command_string, shell=True)



In [ ]:
# @title  ### Visualize Proofreading Outputs
import os
from IPython.display import Image, display
from pathlib import Path

proofreading_out_dir = Path('./proofreading_out/')

if proofreading_out_dir.exists() and proofreading_out_dir.is_dir():
    png_files = reversed(sorted(list(proofreading_out_dir.glob('*.png'))))

    if png_files:
        for png_file in png_files:
            if 'conditional_probs_mean_plus_stddev' in png_file.stem:
              continue
            print(f"Displaying: {png_file.name}")
            display(Image(filename=str(png_file)))
    else:
        print(f"No PNG files found in {proofreading_out_dir}")
else:
    print(f"Directory {proofreading_out_dir} does not exist or is not a directory.")

In [ ]:
# @title ### Download Proofreading Results
import os

# @markdown ### Optionally, don't download anything when cell is run
disable_download = False # @param {"type": "boolean"}

default_zip_output = '/content/lasermpnn_proofreading_outputs.zip'

if not disable_download:
  if os.path.exists(default_zip_output):
    os.remove(default_zip_output)

  subprocess.run(f'zip -j {default_zip_output} /content/proofreading_out/*', shell=True)
  files.download(default_zip_output)
